In [2]:
import os
import mne
# import PyQt6
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from collections import Counter

import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold

from scipy import stats
from scipy.stats import ttest_rel
from scipy.stats import false_discovery_control

# %matplotlib qt

In [3]:
folders = [x for x in os.listdir('OCD/') if os.path.isdir('OCD/' + x) if 'stress' not in x]
fldr2label = {folders[i]: i for i in range(len(folders))}
fldr2label

{'anxiety_ADD (47)': 0,
 'anxiety_AFD (18)': 1,
 'anxiety_general_AD (16)': 2,
 'bipolar_BPD_1(26)': 3,
 'bipolar_BPD_2(25)': 4,
 'controls (157)': 5,
 'Cyclothymia(10)': 6,
 'depression_mild (29)': 7,
 'depression_moderade (47)': 8,
 'depression_severe (32)': 9,
 'personality_disorder (56)': 10}

In [4]:
label2class = {
    0: 0, 1: 0, 2: 0, # anxiety
    3: 1, 4: 1, # bipolar
    5: 2, # control
    6: 3, # cyclothymia
    7: 4, 8: 4, 9: 4, # depression
    10: 5, # personality disorder
}

In [5]:
og_files = []
zg_files = []
og_labels = []
zg_labels = []
for fldr in folders:
    pth =  'OCD/' + fldr
    og_pths = [x for x in os.listdir(pth) if x.lower().endswith('.edf') and ('og.' in x.lower() or 'ог.' in x.lower() or 'eo.' in x.lower() or '_eo' in x.lower())]
    zg_pths = [x for x in os.listdir(pth) if x.lower().endswith('.edf') and ('zg.' in x.lower() or 'зг.' in x.lower() or 'ec.' in x.lower() or 'fon.' in x.lower() or '_ec' in x.lower() or 'eс.' in x.lower())]
    left = [x for x in os.listdir(pth) if x not in og_pths and x not in zg_pths]
    if len(left) > 0:
        print(pth, left)
    for f in og_pths:
        og_files.append(pth + '/' + f)
        og_labels.append(fldr2label[fldr])
    for f in zg_pths:
        zg_files.append(pth + '/' + f)
        zg_labels.append(fldr2label[fldr])

print(f'Number of files for open eyes: {len(og_files)}')
print(f'Number of files for closed eyes: {len(zg_files)}')

# ensured that og_files[i] is the pair for zg_files[i]

Number of files for open eyes: 461
Number of files for closed eyes: 461


# Topography features

In [62]:
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from sklearn.svm import SVC

In [63]:
def get_frontal_features(ch_names, psd):
    indices = [i for i in range(len(ch_names)) if 'F3' in ch_names[i] or 'Fz' in ch_names[i] or 'F4' in ch_names[i]]
    return psd[indices]

def get_central_features(ch_names, psd):
    indices = [i for i in range(len(ch_names)) if 'C3' in ch_names[i] or 'Cz' in ch_names[i] or 'C4' in ch_names[i]]
    return psd[indices]

def get_parietal_features(ch_names, psd):
    indices = [i for i in range(len(ch_names)) if 'P3' in ch_names[i] or 'Pz' in ch_names[i] or 'P4' in ch_names[i]]
    return psd[indices]

def get_frontal2central_gradient(ch_names, psd):
    frontal_indices = [i for i in range(len(ch_names)) if 'F3' in ch_names[i] or 'Fz' in ch_names[i] or 'F4' in ch_names[i]]
    central_indices = [i for i in range(len(ch_names)) if 'C3' in ch_names[i] or 'Cz' in ch_names[i] or 'C4' in ch_names[i]]
    return np.sum(psd[frontal_indices], axis=0) / np.sum(psd[central_indices], axis=0)

def get_frontal2parietal_gradient(ch_names, psd):
    frontal_indices = [i for i in range(len(ch_names)) if 'F3' in ch_names[i] or 'Fz' in ch_names[i] or 'F4' in ch_names[i]]
    parietal_indices = [i for i in range(len(ch_names)) if 'P3' in ch_names[i] or 'Pz' in ch_names[i] or 'P4' in ch_names[i]]
    return np.sum(psd[frontal_indices], axis=0) / np.sum(psd[parietal_indices], axis=0)

def get_central2parietal_gradient(ch_names, psd):
    central_indices = [i for i in range(len(ch_names)) if 'C3' in ch_names[i] or 'Cz' in ch_names[i] or 'C4' in ch_names[i]]
    parietal_indices = [i for i in range(len(ch_names)) if 'P3' in ch_names[i] or 'Pz' in ch_names[i] or 'P4' in ch_names[i]]
    return np.sum(psd[central_indices], axis=0) / np.sum(psd[parietal_indices], axis=0)

def get_front2back_gradient(ch_names, psd):
    front_indices = [i for i in range(len(ch_names)) if any([x in ch_names[i] for x in ['Fp1', 'Fp2', 'F3', 'Fz', 'F4']])]  
    back_indices = [i for i in range(len(ch_names)) if any([x in ch_names[i] for x in ['O1', 'O2', 'P3', 'Pz', 'P4']])]
    return np.sum(psd[front_indices], axis=0) / np.sum(psd[back_indices], axis=0)

def get_left2right_gradient(ch_names, psd):
    left_indices = [i for i in range(len(ch_names)) if any([x in ch_names[i] for x in ['F3', 'C3', 'P3', 'O1']])]
    right_indices = [i for i in range(len(ch_names)) if any([x in ch_names[i] for x in ['F4', 'C4', 'P4', 'O2']])]
    return np.sum(psd[left_indices], axis=0) / np.sum(psd[right_indices], axis=0)


## Open eyes

In [ ]:
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
freq_bands = [(6, 8), (8, 10), (10, 12), (12, 14), (13, 20), (20, 26), (8, 13), (13, 26), (8, 26), (6, 26)]

classifiers = {'SVM': SVC(random_state=92), 'KNN': KNeighborsClassifier()}

topo_features_og = []

for i in tqdm(range(len(og_files))):
    path = og_files[i]
    try:
        sample = mne.io.read_raw_edf(path, verbose=False, preload=True)
    except Exception as e:
        print(f'skipped {path}')
        continue
    # skip faulty data for now
    if 'chan' in sample.ch_names[0].lower():
        print(f'skipped {path}')
        continue
    sample = sample.filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    # get only necessary channels, reorder them
    channels = sample.ch_names
    to_drop = channels[19:]
    sample.drop_channels(to_drop)
    new_idx = []
    skip = False
    for ch in channels2use:
        found = False
        for k in range(19):
            if ch in channels[k]:
                new_idx.append(k)
                found = True
                break
        if not found:
            skip = True
            break
    if skip:
        print(f'skipped {path}')
        continue
    
    s_freq = int(sample.info['sfreq'])
    data = sample.get_data()[new_idx, :int(13 * s_freq)]
    psd, freqs = mne.time_frequency.psd_array_multitaper(data, sfreq=s_freq, fmin=0.5, fmax=30, normalization='length', verbose=False)
    topo_features_og.append({
        'frontal': get_frontal_features(channels, psd),
        'central': get_central_features(channels, psd),
        'parietal': get_parietal_features(channels, psd),
        'frontal2central': get_frontal2central_gradient(channels, psd),
        'frontal2parietal': get_frontal2parietal_gradient(channels, psd),
        'central2parietal': get_central2parietal_gradient(channels, psd),
        'front2back': get_front2back_gradient(channels, psd),
        'left2right': get_left2right_gradient(channels, psd),
        'label': og_labels[i]
    })

  0%|          | 0/461 [00:00<?, ?it/s]

skipped OCD/controls (157)/BORUTTO_JANNA_VLADIMIROVNA_48_EO_free.edf
skipped OCD/controls (157)/Kutuz_f23_contr_og.edf
skipped OCD/controls (157)/MANUILOVA_ELENA_55_og.edf
skipped OCD/controls (157)/Martinenko_m45_og.edf
skipped OCD/controls (157)/Skopincev_20_EO_free.edf
skipped OCD/depression_moderade (47)/FiAV_m50_f32-1_At_Nt_og.edf


In [ ]:
freq_bands = [(6, 8), (8, 10), (10, 12), (12, 14), (13, 20), (20, 26), (8, 13), (13, 26), (8, 26), (6, 26)]

classifiers = {'SVM': SVC(random_state=92), 'KNN': KNeighborsClassifier()}

topo_features_og = []

for i in tqdm(range(len(og_files))):
    path = og_files[i]
    try:
        sample = mne.io.read_raw_edf(path, verbose=False, preload=True)
    except Exception as e:
        print(f'skipped {path}')
        continue
    # skip faulty data for now
    if 'chan' in sample.ch_names[0].lower():
        print(f'skipped {path}')
        continue
    sample = sample.filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    # get only necessary channels, reorder them
    channels = sample.ch_names
    to_drop = channels[19:]
    sample.drop_channels(to_drop)
    new_idx = []
    skip = False
    for ch in channels2use:
        found = False
        for k in range(19):
            if ch in channels[k]:
                new_idx.append(k)
                found = True
                break
        if not found:
            skip = True
            break
    if skip:
        print(f'skipped {path}')
        continue
    
    s_freq = int(sample.info['sfreq'])
    data = sample.get_data()[new_idx, :int(13 * s_freq)]
    psd, freqs = mne.time_frequency.psd_array_multitaper(data, sfreq=s_freq, fmin=0.5, fmax=30, normalization='length', verbose=False)
    topo_features_og.append({
        'frontal': get_frontal_features(channels, psd),
        'central': get_central_features(channels, psd),
        'parietal': get_parietal_features(channels, psd),
        'frontal2central': get_frontal2central_gradient(channels, psd),
        'frontal2parietal': get_frontal2parietal_gradient(channels, psd),
        'central2parietal': get_central2parietal_gradient(channels, psd),
        'front2back': get_front2back_gradient(channels, psd),
        'left2right': get_left2right_gradient(channels, psd),
        'label': og_labels[i]
    })

In [10]:
frontal_features = [x['frontal'] for x in topo_features_og]
central_features = [x['central'] for x in topo_features_og]
parietal_features = [x['parietal'] for x in topo_features_og]
frontal2central_features = [x['frontal2central'] for x in topo_features_og]
frontal2parietal_features = [x['frontal2parietal'] for x in topo_features_og]
central2parietal_features = [x['central2parietal'] for x in topo_features_og]
front2back_features = [x['front2back'] for x in topo_features_og]
left2right_features = [x['left2right'] for x in topo_features_og]
labels = [label2class[x['label']] for x in topo_features_og]

### Features one by one

In [11]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

for k in topo_features_og[0].keys():
    if k == 'label':
        continue
    f1_scores = []
    print(k)
    features = [x[k].flatten() for x in topo_features_og]
    for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
        clf = KNeighborsClassifier()
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(labels)[train]
        y_test = np.array(labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

frontal
	F1 score: 0.2592077975691313, std: 0.0331177315810539
central
	F1 score: 0.2975141702272374, std: 0.06496850930159766
parietal
	F1 score: 0.2714367519691151, std: 0.04056420378907571
frontal2central
	F1 score: 0.2214096426924709, std: 0.03215061714543179
frontal2parietal
	F1 score: 0.22861930587713858, std: 0.04506734424543532
central2parietal
	F1 score: 0.22521152409605474, std: 0.02993434067915106
front2back
	F1 score: 0.22236363319012176, std: 0.07253550453424637
left2right
	F1 score: 0.22111362308900268, std: 0.041923882371954865


In [29]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

for k in topo_features_og[0].keys():
    if k == 'label':
        continue
    f1_scores = []
    print(k)
    features = [x[k].flatten() for x in topo_features_og]
    for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
        clf = SVC(kernel='linear', random_state=92)
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(labels)[train]
        y_test = np.array(labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

frontal
	F1 score: 0.08346655828026915, std: 0.0010066547739388007
central
	F1 score: 0.08346655828026915, std: 0.0010066547739388007
parietal
	F1 score: 0.08346655828026915, std: 0.0010066547739388007
frontal2central
	F1 score: 0.29041750187339743, std: 0.09293462670727914
frontal2parietal
	F1 score: 0.319325395175667, std: 0.06467402117867631
central2parietal
	F1 score: 0.3130902290908547, std: 0.06881299806204907
front2back
	F1 score: 0.30889966043419675, std: 0.030708705634282792
left2right
	F1 score: 0.3120049528015053, std: 0.09034791404089977


In [30]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

for k in topo_features_og[0].keys():
    if k == 'label':
        continue
    f1_scores = []
    print(k)
    features = [x[k].flatten() for x in topo_features_og]
    for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
        clf = SVC(kernel='rbf', random_state=92)
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(labels)[train]
        y_test = np.array(labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

frontal
	F1 score: 0.15120799000661703, std: 0.03624411623139056
central
	F1 score: 0.15316782271491508, std: 0.03212641540549029
parietal
	F1 score: 0.17066426936406295, std: 0.01764352877843459
frontal2central
	F1 score: 0.1804713258059818, std: 0.014315419961446177
frontal2parietal
	F1 score: 0.19724316777534406, std: 0.046640177736604525
central2parietal
	F1 score: 0.24681472812989685, std: 0.05700512204672316
front2back
	F1 score: 0.1614220911976807, std: 0.036681975959625225
left2right
	F1 score: 0.219324585318466, std: 0.020663446681646224


Cyclothymia removed

In [12]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

new_ids = [i for i in range(len(labels)) if labels[i] != 3]
new_labels = [labels[i] for i in new_ids]

for k in topo_features_og[0].keys():
    if k == 'label':
        continue
    f1_scores = []
    print(k)
    features = [topo_features_og[i][k].flatten() for i in new_ids]
    for i, (train, test) in enumerate(kf.split(list(range(len(new_labels))), new_labels)):
        clf = KNeighborsClassifier()
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(new_labels)[train]
        y_test = np.array(new_labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

frontal
	F1 score: 0.3333673420203881, std: 0.015862471402012614
central
	F1 score: 0.30215783879263497, std: 0.02591004527602023
parietal
	F1 score: 0.3001814200908197, std: 0.04358235436450847
frontal2central
	F1 score: 0.24529021393343714, std: 0.058842430640560384
frontal2parietal
	F1 score: 0.2541389061841216, std: 0.015118255122068379
central2parietal
	F1 score: 0.25965921894987876, std: 0.012596087907632067
front2back
	F1 score: 0.2546973989404382, std: 0.04738356991003393
left2right
	F1 score: 0.21499428954536565, std: 0.03885314242948911


In [31]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

new_ids = [i for i in range(len(labels)) if labels[i] != 3]
new_labels = [labels[i] for i in new_ids]

for k in topo_features_og[0].keys():
    if k == 'label':
        continue
    f1_scores = []
    print(k)
    features = [topo_features_og[i][k].flatten() for i in new_ids]
    for i, (train, test) in enumerate(kf.split(list(range(len(new_labels))), new_labels)):
        clf = SVC(kernel='linear', random_state=92)
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(new_labels)[train]
        y_test = np.array(new_labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

frontal
	F1 score: 0.10183753501400561, std: 0.0012213142134885366
central
	F1 score: 0.10183753501400561, std: 0.0012213142134885366
parietal
	F1 score: 0.10183753501400561, std: 0.0012213142134885366
frontal2central
	F1 score: 0.30821627711817745, std: 0.04271249234436142
frontal2parietal
	F1 score: 0.27510560665080597, std: 0.04226229571068722
central2parietal
	F1 score: 0.29239288773688926, std: 0.04809069203558857
front2back
	F1 score: 0.2913740099534804, std: 0.019063868983576565
left2right
	F1 score: 0.32396530752344066, std: 0.051360837039495955


### All features

In [13]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in topo_features_og[0].keys() if k != 'label']))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = KNeighborsClassifier()
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.2255292276372995, std: 0.0432066542587069


In [28]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in topo_features_og[0].keys() if k != 'label']))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.35960690719541166, std: 0.06110422284505228


In [ ]:
# removing any gives no improvement
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in ['frontal', 'central', 'parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = KNeighborsClassifier()
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.3634503399214013, std: 0.059222149304686146


In [35]:
# removing any gives no improvement
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in ['frontal', 'central', 'parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='rbf', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.15058825665012343, std: 0.03763995427724384


In [15]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in ['frontal2central', 'frontal2parietal', 'central2parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = KNeighborsClassifier()
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.21355014049918872, std: 0.061781380363058594


In [37]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in ['frontal2central', 'frontal2parietal', 'central2parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.32009901712917077, std: 0.05568654884733051


In [19]:
# gives no improvement
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in ['frontal', 'central', 'parietal', 'central2parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = KNeighborsClassifier()
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.22521152409605474, std: 0.02993434067915106


In [20]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in ['front2back', 'left2right']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = KNeighborsClassifier()
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.2087081878271031, std: 0.028073682575674042


In [40]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in ['front2back', 'left2right']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.3530919052901495, std: 0.07450495464961822


In [25]:
# gives no improvement
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in ['frontal', 'central', 'parietal', 'front2back']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = KNeighborsClassifier()
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.22236363319012176, std: 0.07253550453424637


In [50]:
# 'frontal2central', 'frontal2parietal', 'central2parietal'
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in ['front2back', 'left2right', 'frontal2parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.37389036853446267, std: 0.10037176650941275


### Best configuration

In [51]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_og[i][k].flatten() for k in ['front2back', 'left2right', 'frontal2parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.37389036853446267, std: 0.10037176650941275


## Closed eyes

In [52]:
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
freq_bands = [(6, 8), (8, 10), (10, 12), (12, 14), (13, 20), (20, 26), (8, 13), (13, 26), (8, 26), (6, 26)]

classifiers = {'SVM': SVC(random_state=92), 'KNN': KNeighborsClassifier()}

topo_features_zg = []

to_skip = ['BORUTTO_JANNA_VLADIMIROVNA', 'Kutuz_f23_contr', 'MANUILOVA_ELENA_55', 'Martinenko_m45', 'Skopincev_20', 'FiAV_m50']

for i in tqdm(range(len(zg_files))):
    path = zg_files[i]
    if any([x in path for x in to_skip]):
        continue
    try:
        sample = mne.io.read_raw_edf(path, verbose=False, preload=True)
    except Exception as e:
        print(f'skipped {path}')
        continue
    # skip faulty data for now
    if 'chan' in sample.ch_names[0].lower():
        print(f'skipped {path}')
        continue
    sample = sample.filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    # get only necessary channels, reorder them
    channels = sample.ch_names
    to_drop = channels[19:]
    sample.drop_channels(to_drop)
    new_idx = []
    skip = False
    for ch in channels2use:
        found = False
        for k in range(19):
            if ch in channels[k]:
                new_idx.append(k)
                found = True
                break
        if not found:
            skip = True
            break
    if skip:
        print(f'skipped {path}')
        continue
    
    data = sample.get_data()[new_idx, :int(13 * s_freq)]
    s_freq = int(sample.info['sfreq'])
    psd, freqs = mne.time_frequency.psd_array_multitaper(data, sfreq=s_freq, fmin=0.5, fmax=30, normalization='length', verbose=False)
    topo_features_zg.append({
        'frontal': get_frontal_features(channels, psd),
        'central': get_central_features(channels, psd),
        'parietal': get_parietal_features(channels, psd),
        'frontal2central': get_frontal2central_gradient(channels, psd),
        'frontal2parietal': get_frontal2parietal_gradient(channels, psd),
        'central2parietal': get_central2parietal_gradient(channels, psd),
        'front2back': get_front2back_gradient(channels, psd),
        'left2right': get_left2right_gradient(channels, psd),
        'label': zg_labels[i]
    })

  0%|          | 0/461 [00:00<?, ?it/s]

In [53]:
frontal_features = [x['frontal'] for x in topo_features_zg]
central_features = [x['central'] for x in topo_features_zg]
parietal_features = [x['parietal'] for x in topo_features_zg]
frontal2central_features = [x['frontal2central'] for x in topo_features_zg]
frontal2parietal_features = [x['frontal2parietal'] for x in topo_features_zg]
central2parietal_features = [x['central2parietal'] for x in topo_features_zg]
front2back_features = [x['front2back'] for x in topo_features_zg]
left2right_features = [x['left2right'] for x in topo_features_zg]
labels = [label2class[x['label']] for x in topo_features_zg]

### Features one by one

In [54]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

for k in topo_features_zg[0].keys():
    if k == 'label':
        continue
    f1_scores = []
    print(k)
    features = [x[k].flatten() for x in topo_features_zg]
    for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
        clf = KNeighborsClassifier()
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(labels)[train]
        y_test = np.array(labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

frontal
	F1 score: 0.30285363888726624, std: 0.06781052929165378
central
	F1 score: 0.2848691147420189, std: 0.02290476699718193
parietal
	F1 score: 0.25648798326282196, std: 0.04065521642726859
frontal2central
	F1 score: 0.24441245105823478, std: 0.05267662490576458
frontal2parietal
	F1 score: 0.2779116969350872, std: 0.07698657747748691
central2parietal
	F1 score: 0.20363025838477453, std: 0.02041883971918726
front2back
	F1 score: 0.27122909354836644, std: 0.1002099318227656
left2right
	F1 score: 0.19824510254730368, std: 0.045638871919688684


In [55]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

for k in topo_features_zg[0].keys():
    if k == 'label':
        continue
    f1_scores = []
    print(k)
    features = [x[k].flatten() for x in topo_features_zg]
    for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
        clf = SVC(kernel='linear', random_state=92)
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(labels)[train]
        y_test = np.array(labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

frontal
	F1 score: 0.08346655828026915, std: 0.0010066547739388007
central
	F1 score: 0.08346655828026915, std: 0.0010066547739388007
parietal
	F1 score: 0.08346655828026915, std: 0.0010066547739388007
frontal2central
	F1 score: 0.323066573366932, std: 0.053091895649082996
frontal2parietal
	F1 score: 0.3330788662654639, std: 0.056277002965905404
central2parietal
	F1 score: 0.304654968575258, std: 0.06675949502957665
front2back
	F1 score: 0.3098296262173758, std: 0.06678490550360089
left2right
	F1 score: 0.3843928498103059, std: 0.07004854611053186


In [56]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

for k in topo_features_zg[0].keys():
    if k == 'label':
        continue
    f1_scores = []
    print(k)
    features = [x[k].flatten() for x in topo_features_zg]
    for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
        clf = SVC(kernel='rbf', random_state=92)
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(labels)[train]
        y_test = np.array(labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

frontal
	F1 score: 0.23658240353175408, std: 0.0639084589925021
central
	F1 score: 0.2221087138246836, std: 0.01886986398410359
parietal
	F1 score: 0.18931848675751145, std: 0.02549898366190406
frontal2central
	F1 score: 0.20423491482607542, std: 0.051769527197590585
frontal2parietal
	F1 score: 0.20591416283201497, std: 0.07551960161352181
central2parietal
	F1 score: 0.1950223760913416, std: 0.05943253098869699
front2back
	F1 score: 0.10371669916493942, std: 0.013002611574796062
left2right
	F1 score: 0.1986338515553709, std: 0.02983804637304659


Cyclothymia removed

In [57]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

new_ids = [i for i in range(len(labels)) if labels[i]  != 3]
new_labels = [labels[i] for i in new_ids]

for k in topo_features_zg[0].keys():
    if k == 'label':
        continue
    f1_scores = []
    print(k)
    features = [topo_features_zg[i][k].flatten() for i in new_ids]
    for i, (train, test) in enumerate(kf.split(list(range(len(new_labels))), new_labels)):
        clf = KNeighborsClassifier()
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(new_labels)[train]
        y_test = np.array(new_labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

frontal
	F1 score: 0.3266197421089034, std: 0.07695552209700622
central
	F1 score: 0.34406477831495735, std: 0.051707747695260965
parietal
	F1 score: 0.3025150425743915, std: 0.05573951008551902
frontal2central
	F1 score: 0.22567084689387565, std: 0.02356654787476238
frontal2parietal
	F1 score: 0.27558643138740935, std: 0.03326890285700438
central2parietal
	F1 score: 0.23699231305056373, std: 0.031144920287603295
front2back
	F1 score: 0.22731341953961506, std: 0.054204315138669086
left2right
	F1 score: 0.22777402468016533, std: 0.035904867054910215


In [58]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

new_ids = [i for i in range(len(labels)) if labels[i]  != 3]
new_labels = [labels[i] for i in new_ids]

for k in topo_features_zg[0].keys():
    if k == 'label':
        continue
    f1_scores = []
    print(k)
    features = [topo_features_zg[i][k].flatten() for i in new_ids]
    for i, (train, test) in enumerate(kf.split(list(range(len(new_labels))), new_labels)):
        clf = SVC(kernel='linear', random_state=92)
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(new_labels)[train]
        y_test = np.array(new_labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

frontal
	F1 score: 0.10183753501400561, std: 0.0012213142134885366
central
	F1 score: 0.10183753501400561, std: 0.0012213142134885366
parietal
	F1 score: 0.10183753501400561, std: 0.0012213142134885366
frontal2central
	F1 score: 0.3077147595715619, std: 0.019422723830278665
frontal2parietal
	F1 score: 0.2947347922473186, std: 0.024525809487056825
central2parietal
	F1 score: 0.29624505785906907, std: 0.051307854992245695
front2back
	F1 score: 0.262032767789029, std: 0.03425948332070872
left2right
	F1 score: 0.3522216470284893, std: 0.043370779515096596


### All features

In [59]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_zg[i][k].flatten() for k in topo_features_zg[0].keys() if k != 'label']))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = KNeighborsClassifier()
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.24414065882132663, std: 0.03862376404354972


In [60]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_zg[i][k].flatten() for k in topo_features_zg[0].keys() if k != 'label']))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.37234690454583946, std: 0.06896053111586559


In [61]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_zg[i][k].flatten() for k in ['frontal', 'central', 'parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = KNeighborsClassifier()
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.3053411699240857, std: 0.052865554209587585


In [62]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_zg[i][k].flatten() for k in ['frontal', 'central', 'parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.08346655828026915, std: 0.0010066547739388007


In [63]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_zg[i][k].flatten() for k in ['frontal2central', 'frontal2parietal', 'central2parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = KNeighborsClassifier()
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.2330749023879152, std: 0.0697461302683816


In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_zg[i][k].flatten() for k in ['frontal2central', 'frontal2parietal', 'central2parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.33214438559552373, std: 0.07802547853908635


In [71]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_zg[i][k].flatten() for k in ['front2back', 'left2right']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = KNeighborsClassifier()
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.2677898250824356, std: 0.08006467208658546


In [74]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_zg[i][k].flatten() for k in ['front2back', 'left2right']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.3803941117421572, std: 0.06041477206549373


In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_zg[i][k].flatten() for k in ['front2back', 'left2right', 'frontal2parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.38375405167808957, std: 0.0631613582191986


### Best configuration

In [80]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([topo_features_zg[i][k].flatten() for k in ['front2back', 'left2right', 'frontal2parietal']]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

F1 score: 0.38375405167808957, std: 0.0631613582191986


## Silly experiment for frequency bands selection

Use the best configuration: SVM, with `['front2back', 'left2right', 'frontal2parietal']` features, calculate them in the frequency bands and assess the classification quality

### Open eyes

In [7]:
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
freq_bands = [(6, 8), (8, 10), (10, 12), (12, 14), (13, 20), (20, 26), (8, 13), (13, 26), (8, 26), (6, 26)]

classifiers = {'SVM': SVC(random_state=92), 'KNN': KNeighborsClassifier()}

band2features = {fb: [] for fb in freq_bands}

for i in tqdm(range(len(og_files))):
    path = og_files[i]
    try:
        sample = mne.io.read_raw_edf(path, verbose=False, preload=True)
    except Exception as e:
        print(f'skipped {path}')
        continue
    # skip faulty data for now
    if 'chan' in sample.ch_names[0].lower():
        print(f'skipped {path}')
        continue
    sample = sample.filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    # get only necessary channels, reorder them
    channels = sample.ch_names
    to_drop = channels[19:]
    sample.drop_channels(to_drop)
    new_idx = []
    skip = False
    for ch in channels2use:
        found = False
        for k in range(19):
            if ch in channels[k]:
                new_idx.append(k)
                found = True
                break
        if not found:
            skip = True
            break
    if skip:
        print(f'skipped {path}')
        continue
    
    s_freq = int(sample.info['sfreq'])
    data = sample.get_data()[new_idx, :int(13 * s_freq)]

    for freq_band in freq_bands:
        psd, freqs = mne.time_frequency.psd_array_multitaper(data, sfreq=s_freq, fmin=freq_band[0], fmax=freq_band[1], normalization='length', verbose=False)
        band2features[freq_band].append({
            'frontal2parietal': get_frontal2parietal_gradient(channels, psd),
            'front2back': get_front2back_gradient(channels, psd),
            'left2right': get_left2right_gradient(channels, psd),
            'label': og_labels[i]
        })

for freq_band in tqdm(freq_bands):
    frontal2parietal_features = [x['frontal2parietal'] for x in band2features[freq_band]]
    front2back_features = [x['front2back'] for x in band2features[freq_band]]
    left2right_features = [x['left2right'] for x in band2features[freq_band]]
    labels = [label2class[x['label']] for x in band2features[freq_band]]

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

    features = []
    for i in range(len(labels)):
        features.append(np.concatenate([band2features[freq_band][i][k].flatten() for k in ['front2back', 'left2right', 'frontal2parietal']]))
    f1_scores = []
    for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
        clf = SVC(kernel='linear', random_state=92)
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(labels)[train]
        y_test = np.array(labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'Frequency band: {freq_band}')
    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

  0%|          | 0/461 [00:00<?, ?it/s]

skipped OCD/controls (157)/BORUTTO_JANNA_VLADIMIROVNA_48_EO_free.edf
skipped OCD/controls (157)/Kutuz_f23_contr_og.edf
skipped OCD/controls (157)/MANUILOVA_ELENA_55_og.edf
skipped OCD/controls (157)/Martinenko_m45_og.edf
skipped OCD/controls (157)/Skopincev_20_EO_free.edf
skipped OCD/depression_moderade (47)/FiAV_m50_f32-1_At_Nt_og.edf


  0%|          | 0/10 [00:00<?, ?it/s]

Frequency band: (6, 8)
	F1 score: 0.26848113206444474, std: 0.07085294974387475
Frequency band: (8, 10)
	F1 score: 0.20783323174385798, std: 0.019662928400449767
Frequency band: (10, 12)
	F1 score: 0.3047224872645723, std: 0.07045955229368081
Frequency band: (12, 14)
	F1 score: 0.27835118490906585, std: 0.0896977539564451
Frequency band: (13, 20)
	F1 score: 0.3263384260116519, std: 0.09769666423557156
Frequency band: (20, 26)
	F1 score: 0.29559537153694265, std: 0.051315619861936286
Frequency band: (8, 13)
	F1 score: 0.30897636707247617, std: 0.07061714493175168
Frequency band: (13, 26)
	F1 score: 0.3046687907366075, std: 0.07472268845345822
Frequency band: (8, 26)
	F1 score: 0.31903753478088503, std: 0.11975594287555774
Frequency band: (6, 26)
	F1 score: 0.3483562450686862, std: 0.09520754049442134


### Closed eyes

In [ ]:
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
freq_bands = [(6, 8), (8, 10), (10, 12), (12, 14), (13, 20), (20, 26), (8, 13), (13, 26), (8, 26), (6, 26)]

classifiers = {'SVM': SVC(random_state=92), 'KNN': KNeighborsClassifier()}

band2features = {fb: [] for fb in freq_bands}

to_skip = ['BORUTTO_JANNA_VLADIMIROVNA', 'Kutuz_f23_contr', 'MANUILOVA_ELENA_55', 'Martinenko_m45', 'Skopincev_20', 'FiAV_m50']

for i in tqdm(range(len(zg_files))):
    path = zg_files[i]
    if any([x in path for x in to_skip]):
        continue
    try:
        sample = mne.io.read_raw_edf(path, verbose=False, preload=True)
    except Exception as e:
        print(f'skipped {path}')
        continue
    # skip faulty data for now
    if 'chan' in sample.ch_names[0].lower():
        print(f'skipped {path}')
        continue
    sample = sample.filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    # get only necessary channels, reorder them
    channels = sample.ch_names
    to_drop = channels[19:]
    sample.drop_channels(to_drop)
    new_idx = []
    skip = False
    for ch in channels2use:
        found = False
        for k in range(19):
            if ch in channels[k]:
                new_idx.append(k)
                found = True
                break
        if not found:
            skip = True
            break
    if skip:
        print(f'skipped {path}')
        continue
    
    s_freq = int(sample.info['sfreq'])
    data = sample.get_data()[new_idx, :int(13 * s_freq)]

    for freq_band in freq_bands:
        psd, freqs = mne.time_frequency.psd_array_multitaper(data, sfreq=s_freq, fmin=freq_band[0], fmax=freq_band[1], normalization='length', verbose=False)
        band2features[freq_band].append({
            'frontal2parietal': get_frontal2parietal_gradient(channels, psd),
            'front2back': get_front2back_gradient(channels, psd),
            'left2right': get_left2right_gradient(channels, psd),
            'label': zg_labels[i]
        })

for freq_band in tqdm(freq_bands):
    frontal2parietal_features = [x['frontal2parietal'] for x in band2features[freq_band]]
    front2back_features = [x['front2back'] for x in band2features[freq_band]]
    left2right_features = [x['left2right'] for x in band2features[freq_band]]
    labels = [label2class[x['label']] for x in band2features[freq_band]]

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

    features = []
    for i in range(len(labels)):
        features.append(np.concatenate([band2features[freq_band][i][k].flatten() for k in ['front2back', 'left2right', 'frontal2parietal']]))
    f1_scores = []
    for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
        clf = SVC(kernel='linear', random_state=92)
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(labels)[train]
        y_test = np.array(labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'Frequency band: {freq_band}')
    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

  0%|          | 0/461 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

Frequency band: (6, 8)
	F1 score: 0.30307584735639803, std: 0.05995196035956741
Frequency band: (8, 10)
	F1 score: 0.2840928244085633, std: 0.04880321884995467
Frequency band: (10, 12)
	F1 score: 0.2940754143882779, std: 0.06772008260566681
Frequency band: (12, 14)
	F1 score: 0.2931231772550572, std: 0.06449007885886324
Frequency band: (13, 20)
	F1 score: 0.34541617685485104, std: 0.09123834584447228
Frequency band: (20, 26)
	F1 score: 0.3271764170040032, std: 0.055454677260035286
Frequency band: (8, 13)
	F1 score: 0.34672142042558196, std: 0.043018360548655085
Frequency band: (13, 26)
	F1 score: 0.39079922423820806, std: 0.09121594538736508
Frequency band: (8, 26)
	F1 score: 0.38738379274176704, std: 0.0842138007351277
Frequency band: (6, 26)
	F1 score: 0.37591543817431133, std: 0.07570669571032938


In [21]:
freq_bands = [(8, 13), (13, 26)]
num_samples = len(band2features[freq_bands[0]])
frontal2parietal_features = [np.concatenate([band2features[freq_bands[0]][i]['frontal2parietal'], band2features[freq_bands[1]][i]['frontal2parietal']]) for i in range(num_samples)]
front2back_features = [np.concatenate([band2features[freq_bands[0]][i]['front2back'], band2features[freq_bands[1]][i]['front2back']]) for i in range(num_samples)]
left2right_features = [np.concatenate([band2features[freq_bands[0]][i]['left2right'], band2features[freq_bands[1]][i]['left2right']]) for i in range(num_samples)]
labels = [label2class[x['label']] for x in band2features[freq_bands[0]]]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([frontal2parietal_features[i], front2back_features[i], left2right_features[i]]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

# print(f'Frequency band: {freq_band}')
print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

	F1 score: 0.39175820193604116, std: 0.08270812591235416


## Diff between open and closed eyes

In [46]:
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
freq_bands = [(6, 8), (8, 10), (10, 12), (12, 14), (13, 20), (20, 26), (8, 13), (13, 26), (8, 26), (6, 26)]

band2features = {fb: [] for fb in freq_bands}

for id in tqdm(range(len(og_files))):
    try:
        sample_og = mne.io.read_raw_edf(og_files[id], verbose=False, preload=True).filter(l_freq=1, h_freq=30, method='iir', verbose=False)
        sample_zg = mne.io.read_raw_edf(zg_files[id], verbose=False, preload=True).filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    except Exception as e:
        print(f'skipped {og_files[id]}')
        continue

    channels_og = sample_og.ch_names
    channels_zg = sample_zg.ch_names
    new_idx_og = []
    new_idx_zg = []
    skip = False
    for ch in channels2use:
        found_og = False
        found_zg = False
        for k in range(19):
            if ch in channels_og[k]:
                new_idx_og.append(k)
                found_og = True
            if ch in channels_zg[k]:
                new_idx_zg.append(k)
                found_zg = True
            if found_og and found_zg:
                break
        if not (found_og and found_zg):
            skip = True
            break
    if skip:
        print(f'skipped {path}')
        continue

    s_freq = int(sample.info['sfreq'])
    data_og = sample_og.get_data()[new_idx_og, :int(13 * s_freq)]
    data_zg = sample_zg.get_data()[new_idx_zg, :int(13 * s_freq)]

    for freq_band in freq_bands:
        psd_og, freqs_og = mne.time_frequency.psd_array_multitaper(data_og, sfreq=s_freq, fmin=freq_band[0], fmax=freq_band[1], normalization='length', verbose=False)
        psd_zg, freqs_zg = mne.time_frequency.psd_array_multitaper(data_zg, sfreq=s_freq, fmin=freq_band[0], fmax=freq_band[1], normalization='length', verbose=False)
        psd_og = psd_og.flatten()
        psd_zg = psd_zg.flatten()

        band2features[freq_band].append({
            'diff': (psd_zg - psd_og) / (psd_zg + psd_og),
            'label': zg_labels[id]
        })

  0%|          | 0/461 [00:00<?, ?it/s]

skipped OCD/personality_disorder (56)/ИОДКОВСКАЯ ОЛЬГА 14 Ф60-3_zg.EDF
skipped OCD/controls (157)/Kutuz_f23_contr_og.edf
skipped OCD/personality_disorder (56)/ИОДКОВСКАЯ ОЛЬГА 14 Ф60-3_zg.EDF
skipped OCD/personality_disorder (56)/ИОДКОВСКАЯ ОЛЬГА 14 Ф60-3_zg.EDF
skipped OCD/controls (157)/Skopincev_20_EO_free.edf
skipped OCD/personality_disorder (56)/ИОДКОВСКАЯ ОЛЬГА 14 Ф60-3_zg.EDF


In [47]:
for freq_band in tqdm(freq_bands):
    features = [x['diff'] for x in band2features[freq_band]]
    labels = [label2class[x['label']] for x in band2features[freq_band]]

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

    f1_scores = []
    for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
        clf = SVC(kernel='linear', random_state=92)
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(labels)[train]
        y_test = np.array(labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    print(f'Frequency band: {freq_band}')
    print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

  0%|          | 0/10 [00:00<?, ?it/s]

Frequency band: (6, 8)
	F1 score: 0.33537955010787707, std: 0.09867617295678782
Frequency band: (8, 10)
	F1 score: 0.34304390469819696, std: 0.06534879389379185
Frequency band: (10, 12)
	F1 score: 0.30385632413596775, std: 0.06920551008140377
Frequency band: (12, 14)
	F1 score: 0.35891938876277835, std: 0.05408068543063446
Frequency band: (13, 20)
	F1 score: 0.3157682467501807, std: 0.08271382775645328
Frequency band: (20, 26)
	F1 score: 0.32112771515323024, std: 0.09984182711851321
Frequency band: (8, 13)
	F1 score: 0.3593487768495748, std: 0.05943226781703187
Frequency band: (13, 26)
	F1 score: 0.32573453983874656, std: 0.09152835891329919
Frequency band: (8, 26)
	F1 score: 0.3533120376265453, std: 0.09426636065645873
Frequency band: (6, 26)
	F1 score: 0.3575293566518568, std: 0.09203107651606571


In [ ]:
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
features_list = []

for id in tqdm(range(len(og_files))):
    try:
        sample_og = mne.io.read_raw_edf(og_files[id], verbose=False, preload=True).filter(l_freq=1, h_freq=30, method='iir', verbose=False)
        sample_zg = mne.io.read_raw_edf(zg_files[id], verbose=False, preload=True).filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    except Exception as e:
        print(f'skipped {og_files[id]}')
        continue

    channels_og = sample_og.ch_names
    channels_zg = sample_zg.ch_names
    new_idx_og = []
    new_idx_zg = []
    skip = False
    for ch in channels2use:
        found_og = False
        found_zg = False
        for k in range(19):
            if ch in channels_og[k]:
                new_idx_og.append(k)
                found_og = True
            if ch in channels_zg[k]:
                new_idx_zg.append(k)
                found_zg = True
            if found_og and found_zg:
                break
        if not (found_og and found_zg):
            skip = True
            break
    if skip:
        print(f'skipped {path}')
        continue

    s_freq = int(sample.info['sfreq'])
    data_og = sample_og.get_data()[new_idx_og, :int(13 * s_freq)]
    data_zg = sample_zg.get_data()[new_idx_zg, :int(13 * s_freq)]

    psd_og, freqs_og = mne.time_frequency.psd_array_multitaper(data_og, sfreq=s_freq, fmin=1, fmax=30, normalization='length', verbose=False)
    psd_zg, freqs_zg = mne.time_frequency.psd_array_multitaper(data_zg, sfreq=s_freq, fmin=1, fmax=30, normalization='length', verbose=False)
    psd_og = psd_og.flatten()
    psd_zg = psd_zg.flatten()

    features_list.append({
        'diff': (psd_zg - psd_og) / (psd_zg + psd_og),
        'label': zg_labels[id]
    })

features = [x['diff'] for x in features_list]
labels = [label2class[x['label']] for x in features_list]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

print(f'F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

  0%|          | 0/461 [00:00<?, ?it/s]

skipped OCD/personality_disorder (56)/ИОДКОВСКАЯ ОЛЬГА 14 Ф60-3_zg.EDF
skipped OCD/controls (157)/Kutuz_f23_contr_og.edf
skipped OCD/personality_disorder (56)/ИОДКОВСКАЯ ОЛЬГА 14 Ф60-3_zg.EDF
skipped OCD/personality_disorder (56)/ИОДКОВСКАЯ ОЛЬГА 14 Ф60-3_zg.EDF
skipped OCD/controls (157)/Skopincev_20_EO_free.edf
skipped OCD/personality_disorder (56)/ИОДКОВСКАЯ ОЛЬГА 14 Ф60-3_zg.EDF
	F1 score: 0.3597074090154452, std: 0.08096837636986069


# Combine diff and closed eyes features

In [73]:
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
freq_bands = [(8, 13), (13, 26)]

band2features = {fb: [] for fb in freq_bands}
band2features[(8, 26)] = []

to_skip = ['BORUTTO_JANNA_VLADIMIROVNA', 'Kutuz_f23_contr', 'MANUILOVA_ELENA_55', 'Martinenko_m45', 'Skopincev_20', 'FiAV_m50']

for i in tqdm(range(len(zg_files))):
    path = zg_files[i]
    if any([x in path for x in to_skip]):
        continue
    try:
        sample_zg = mne.io.read_raw_edf(path, verbose=False, preload=True).filter(l_freq=1, h_freq=30, method='iir', verbose=False)
        sample_og = mne.io.read_raw_edf(og_files[i], verbose=False, preload=True).filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    except Exception as e:
        print(f'skipped {path}')
        continue
    # skip faulty data for now
    if 'chan' in sample.ch_names[0].lower():
        print(f'skipped {path}')
        continue
    # get only necessary channels, reorder them
    cchannels_og = sample_og.ch_names
    channels_zg = sample_zg.ch_names
    new_idx_og = []
    new_idx_zg = []
    skip = False
    for ch in channels2use:
        found_og = False
        found_zg = False
        for k in range(19):
            if ch in channels_og[k]:
                new_idx_og.append(k)
                found_og = True
            if ch in channels_zg[k]:
                new_idx_zg.append(k)
                found_zg = True
            if found_og and found_zg:
                break
        if not (found_og and found_zg):
            skip = True
            break
    if skip:
        print(f'skipped {path}')
        continue
    
    s_freq = int(sample_og.info['sfreq'])
    data_og = sample_og.get_data()[new_idx_og, :int(13 * s_freq)]
    data_zg = sample_zg.get_data()[new_idx_zg, :int(13 * s_freq)]

    for freq_band in freq_bands:
        psd, freqs = mne.time_frequency.psd_array_multitaper(data_zg, sfreq=s_freq, fmin=freq_band[0], fmax=freq_band[1], normalization='length', verbose=False)
        psd_og, freqs_og = mne.time_frequency.psd_array_multitaper(data_og, sfreq=s_freq, fmin=freq_band[0], fmax=freq_band[1], normalization='length', verbose=False)
        band2features[freq_band].append({
            'frontal2parietal': get_frontal2parietal_gradient(channels, psd),
            'front2back': get_front2back_gradient(channels, psd),
            'left2right': get_left2right_gradient(channels, psd),
            'diff': (psd - psd_og) / (psd + psd_og),
            'label': zg_labels[i]
        })
    psd, freqs = mne.time_frequency.psd_array_multitaper(data_zg, sfreq=s_freq, fmin=8, fmax=26, normalization='length', verbose=False)
    psd_og, freqs_og = mne.time_frequency.psd_array_multitaper(data_og, sfreq=s_freq, fmin=8, fmax=26, normalization='length', verbose=False)
    band2features[(8, 26)].append({
        'diff': (psd - psd_og) / (psd + psd_og),
        'label': zg_labels[i]
    })
    

num_samples = len(band2features[freq_bands[0]])
frontal2parietal_features = [np.concatenate([band2features[freq_bands[0]][i]['frontal2parietal'], band2features[freq_bands[1]][i]['frontal2parietal']]) for i in range(num_samples)]
front2back_features = [np.concatenate([band2features[freq_bands[0]][i]['front2back'], band2features[freq_bands[1]][i]['front2back']]) for i in range(num_samples)]
left2right_features = [np.concatenate([band2features[freq_bands[0]][i]['left2right'], band2features[freq_bands[1]][i]['left2right']]) for i in range(num_samples)]
diff_features = [band2features[(8, 26)][i]['diff'].flatten() for i in range(num_samples)]
labels = [label2class[x['label']] for x in band2features[freq_bands[0]]]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([frontal2parietal_features[i], front2back_features[i], left2right_features[i], diff_features[i]]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

# print(f'Frequency band: {freq_band}')
print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

  0%|          | 0/461 [00:00<?, ?it/s]

	F1 score: 0.4032854779349996, std: 0.05853331928560046


In [69]:
band2features[(8, 13)][0]['diff'].shape

(19, 66)

In [70]:
band2features[(13, 26)][0]['diff'].shape

(19, 170)

# Other features

In [53]:
import pickle
from scipy.signal import hilbert
from scipy.stats import entropy
from numpy.linalg import svd
from hurst import compute_Hc
from nolds import lyap_r, corr_dim, sampen
from scipy.spatial.distance import pdist, squareform
from joblib import Parallel, delayed

In [50]:
def hjorth_parameters(data):
    activity = np.var(data)
    mobility = np.sqrt(np.var(np.diff(data)) / activity)
    complexity = np.sqrt(np.var(np.diff(np.diff(data))) / np.var(np.diff(data))) / mobility
    return activity, mobility, complexity

def fractal_dimension(data, k=2):
    n = len(data)
    fd = np.zeros(n - k)
    for i in range(k, n):
        fd[i - k] = np.log(np.std(data[:i])) / np.log(i)
    return np.mean(fd)

def hurst_exponent(data):
    H, _, _ = compute_Hc(data, kind='random_walk')
    return H

def lyapunov_exponent(data, tau=1, embedding_dim=2):
    N = len(data)
    max_index = N - (embedding_dim - 1) * tau
    embedded = np.array([data[i: max_index + i : tau] for i in range(embedding_dim)]).T
    dist_matrix = squareform(pdist(embedded))
    np.fill_diagonal(dist_matrix, np.inf)
    min_indices = np.argmin(dist_matrix, axis=1)
    divergence = np.mean(np.abs(embedded - embedded[min_indices]), axis=0)
    divergence = divergence[divergence > 0]
    log_divergence = np.log(divergence)
    time = np.arange(len(log_divergence))
    slope, _ = np.polyfit(time, log_divergence, 1)
    
    return slope

def correlation_dimension(data):
    return corr_dim(data, 2)

def approximate_entropy(data, m=2, r=0.2):
    return sampen(data, m)

def envelope_mean_frequency(data, sfreq):
    analytic_signal = hilbert(data)
    envelope = np.abs(analytic_signal)
    return np.mean(envelope)

# def extract_features(eeg_data, sfreq):
#     features = []
#     for channel in eeg_data:
#         fd = fractal_dimension(channel)
#         cd = correlation_dimension(channel)
#         le = lyapunov_exponent(channel)
#         he = hurst_exponent(channel)
#         ap_en = approximate_entropy(channel)
#         hjorth_act, hjorth_mob, hjorth_comp = hjorth_parameters(channel)
#         emf = envelope_mean_frequency(channel, sfreq)
        
#         features.append([fd, cd, le, he, ap_en, hjorth_act, hjorth_mob, hjorth_comp, emf])
#     return np.array(features)

def compute_features(channel, sfreq):
    return [
        fractal_dimension(channel),
        correlation_dimension(channel),
        lyapunov_exponent(channel),
        hurst_exponent(channel),
        approximate_entropy(channel),
        *hjorth_parameters(channel),
        envelope_mean_frequency(channel, sfreq)
    ]


def extract_features(eeg_data, sfreq, n_jobs=-1):
    features = Parallel(n_jobs=n_jobs)(delayed(compute_features)(channel, sfreq) for channel in eeg_data)
    return np.array(features)

## Open eyes

In [54]:
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
# freq_bands = [(6, 8), (8, 10), (10, 12), (12, 14), (13, 20), (20, 26), (8, 13), (13, 26), (8, 26), (6, 26)]
# freq_bands = [(8, 13), (13, 26), (6, 26)]
freq_bands = [(1, 30)]

# classifiers = {'SVM': SVC(random_state=92), 'KNN': KNeighborsClassifier()}

band2features_og = {freq: [] for freq in freq_bands}

for i in tqdm(range(len(og_files))):
    path = og_files[i]
    try:
        sample = mne.io.read_raw_edf(path, verbose=False, preload=True)
    except Exception as e:
        print(f'skipped {path}')
        continue
    # skip faulty data for now
    if 'chan' in sample.ch_names[0].lower():
        print(f'skipped {path}')
        continue
    sample = sample.filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    # get only necessary channels, reorder them
    channels = sample.ch_names
    to_drop = channels[19:]
    sample.drop_channels(to_drop)
    new_idx = []
    skip = False
    for ch in channels2use:
        found = False
        for k in range(19):
            if ch in channels[k]:
                new_idx.append(k)
                found = True
                break
        if not found:
            skip = True
            break
    if skip:
        print(f'skipped {path}')
        continue
    
    s_freq = int(sample.info['sfreq'])
    for freq in freq_bands:
        new_sample = sample.copy().filter(l_freq=freq[0], h_freq=freq[1], method='iir', verbose=False)
        data = new_sample.get_data()[new_idx, :10 * s_freq]
        band2features_og[freq].append({
            'features': extract_features(data, s_freq).flatten(),
            'label': og_labels[i]
        })
    with open('other_features_og.pkl', 'wb') as f:
        pickle.dump(band2features_og, f)

  0%|          | 0/461 [00:00<?, ?it/s]

skipped OCD/controls (157)/BORUTTO_JANNA_VLADIMIROVNA_48_EO_free.edf
skipped OCD/controls (157)/Kutuz_f23_contr_og.edf
skipped OCD/controls (157)/MANUILOVA_ELENA_55_og.edf
skipped OCD/controls (157)/Martinenko_m45_og.edf
skipped OCD/controls (157)/Skopincev_20_EO_free.edf
skipped OCD/depression_moderade (47)/FiAV_m50_f32-1_At_Nt_og.edf


In [57]:
for freq_band in tqdm(freq_bands):
    labels = [label2class[x['label']] for x in band2features_og[freq_band]]

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

    print(f'Frequency band: {freq_band}')
    for i in range(9):
        features = [x['features'][19 * i: 19 * (i + 1)] for x in band2features_og[freq_band]]
        f1_scores = []
        for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
            clf = SVC(kernel='linear', random_state=92)
            X_train = np.array(features)[train]
            X_test = np.array(features)[test]
            y_train = np.array(labels)[train]
            y_test = np.array(labels)[test]
            clf.fit(X_train, y_train)   
            f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

        
        print(f'\t{i}: F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

  0%|          | 0/1 [00:00<?, ?it/s]

Frequency band: (1, 30)
	4: F1 score: 0.13248265070001766, std: 0.01742798896132304
	4: F1 score: 0.12570739003893072, std: 0.01609588531686472
	4: F1 score: 0.1265848286517429, std: 0.018245521676266196
	4: F1 score: 0.1044517793627526, std: 0.014862655504165328
	4: F1 score: 0.08346655828026915, std: 0.0010066547739388007
	4: F1 score: 0.08346655828026915, std: 0.0010066547739388007
	4: F1 score: 0.08346655828026915, std: 0.0010066547739388007
	4: F1 score: 0.08346655828026915, std: 0.0010066547739388007
	4: F1 score: 0.08346655828026915, std: 0.0010066547739388007


## Closed eyes

In [58]:
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
# freq_bands = [(6, 8), (8, 10), (10, 12), (12, 14), (13, 20), (20, 26), (8, 13), (13, 26), (8, 26), (6, 26)]
# freq_bands = [(8, 13), (13, 26), (6, 26)]
freq_bands = [(1, 30)]

# classifiers = {'SVM': SVC(random_state=92), 'KNN': KNeighborsClassifier()}

band2features_zg = {freq: [] for freq in freq_bands}
to_skip = ['BORUTTO_JANNA_VLADIMIROVNA', 'Kutuz_f23_contr', 'MANUILOVA_ELENA_55', 'Martinenko_m45', 'Skopincev_20', 'FiAV_m50']

for i in tqdm(range(len(zg_files))):
    path = zg_files[i]
    if any([x in path for x in to_skip]):
        continue
    try:
        sample = mne.io.read_raw_edf(path, verbose=False, preload=True)
    except Exception as e:
        print(f'skipped {path}')
        continue
    # skip faulty data for now
    if 'chan' in sample.ch_names[0].lower():
        print(f'skipped {path}')
        continue
    sample = sample.filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    # get only necessary channels, reorder them
    channels = sample.ch_names
    to_drop = channels[19:]
    sample.drop_channels(to_drop)
    new_idx = []
    skip = False
    for ch in channels2use:
        found = False
        for k in range(19):
            if ch in channels[k]:
                new_idx.append(k)
                found = True
                break
        if not found:
            skip = True
            break
    if skip:
        print(f'skipped {path}')
        continue
    
    s_freq = int(sample.info['sfreq'])
    for freq in freq_bands:
        new_sample = sample.copy().filter(l_freq=freq[0], h_freq=freq[1], method='iir', verbose=False)
        data = new_sample.get_data()[new_idx, :10 * s_freq]
        band2features_zg[freq].append({
            'features': extract_features(data, s_freq).flatten(),
            'label': zg_labels[i]
        })
    with open('other_features_zg.pkl', 'wb') as f:
        pickle.dump(band2features_zg, f)

  0%|          | 0/461 [00:00<?, ?it/s]

In [59]:
for freq_band in tqdm(freq_bands):
    labels = [label2class[x['label']] for x in band2features_zg[freq_band]]

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

    print(f'Frequency band: {freq_band}')
    features = [x['features'] for x in band2features_zg[freq_band]]
    f1_scores = []
    for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
        clf = SVC(kernel='linear', random_state=92)
        X_train = np.array(features)[train]
        X_test = np.array(features)[test]
        y_train = np.array(labels)[train]
        y_test = np.array(labels)[test]
        clf.fit(X_train, y_train)   
        f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

    
    print(f'\t{i}: F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

  0%|          | 0/1 [00:00<?, ?it/s]

Frequency band: (1, 30)
	4: F1 score: 0.19710452175757837, std: 0.012373742205965405


In [60]:
for freq_band in tqdm(freq_bands):
    labels = [label2class[x['label']] for x in band2features_zg[freq_band]]

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

    print(f'Frequency band: {freq_band}')
    for i in range(9):
        features = [x['features'][19 * i: 19 * (i + 1)] for x in band2features_zg[freq_band]]
        f1_scores = []
        for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
            clf = SVC(kernel='linear', random_state=92)
            X_train = np.array(features)[train]
            X_test = np.array(features)[test]
            y_train = np.array(labels)[train]
            y_test = np.array(labels)[test]
            clf.fit(X_train, y_train)   
            f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

        
        print(f'\t{i}: F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

  0%|          | 0/1 [00:00<?, ?it/s]

Frequency band: (1, 30)
	4: F1 score: 0.0938664722087182, std: 0.013883340101509383
	4: F1 score: 0.08360655737704917, std: 0.001198733194334556
	4: F1 score: 0.08388429752066115, std: 0.0012624175468197846
	4: F1 score: 0.09600550964187328, std: 0.0152861536077191
	4: F1 score: 0.09586551054509326, std: 0.015385531154876187
	4: F1 score: 0.08360655737704917, std: 0.001198733194334556
	4: F1 score: 0.09810881433207637, std: 0.012557611825742972
	4: F1 score: 0.08346655828026915, std: 0.0010066547739388007
	4: F1 score: 0.10612283153530204, std: 0.02476920962047959


In [79]:
freq_band = (1, 30)
labels = [label2class[x['label']] for x in band2features_zg[freq_band]]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

print(f'Frequency band: {freq_band}')
features = [np.concatenate([x['features'][19 * 8: 19 * 9], x['features'][19 * 4: 19 * 5]]) for x in band2features_zg[freq_band]]
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))


print(f'\t{i}: F1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

Frequency band: (1, 30)
	4: F1 score: 0.12013656977942692, std: 0.02526280883505698


# Combine all?

In [80]:
channels2use = ['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F7', 'F8', 'T3', 'T4', 'C3', 'Cz', 'C4', 'T5', 'T6', 'P3', 'Pz', 'P4', 'O1', 'O2']
freq_bands = [(8, 13), (13, 26)]

band2features = {fb: [] for fb in freq_bands}
band2features[(8, 26)] = []

to_skip = ['BORUTTO_JANNA_VLADIMIROVNA', 'Kutuz_f23_contr', 'MANUILOVA_ELENA_55', 'Martinenko_m45', 'Skopincev_20', 'FiAV_m50']

for i in tqdm(range(len(zg_files))):
    path = zg_files[i]
    if any([x in path for x in to_skip]):
        continue
    try:
        sample_zg = mne.io.read_raw_edf(path, verbose=False, preload=True).filter(l_freq=1, h_freq=30, method='iir', verbose=False)
        sample_og = mne.io.read_raw_edf(og_files[i], verbose=False, preload=True).filter(l_freq=1, h_freq=30, method='iir', verbose=False)
    except Exception as e:
        print(f'skipped {path}')
        continue
    # skip faulty data for now
    if 'chan' in sample.ch_names[0].lower():
        print(f'skipped {path}')
        continue
    # get only necessary channels, reorder them
    cchannels_og = sample_og.ch_names
    channels_zg = sample_zg.ch_names
    new_idx_og = []
    new_idx_zg = []
    skip = False
    for ch in channels2use:
        found_og = False
        found_zg = False
        for k in range(19):
            if ch in channels_og[k]:
                new_idx_og.append(k)
                found_og = True
            if ch in channels_zg[k]:
                new_idx_zg.append(k)
                found_zg = True
            if found_og and found_zg:
                break
        if not (found_og and found_zg):
            skip = True
            break
    if skip:
        print(f'skipped {path}')
        continue
    
    s_freq = int(sample_og.info['sfreq'])
    data_og = sample_og.get_data()[new_idx_og, :int(13 * s_freq)]
    data_zg = sample_zg.get_data()[new_idx_zg, :int(13 * s_freq)]

    for freq_band in freq_bands:
        psd, freqs = mne.time_frequency.psd_array_multitaper(data_zg, sfreq=s_freq, fmin=freq_band[0], fmax=freq_band[1], normalization='length', verbose=False)
        psd_og, freqs_og = mne.time_frequency.psd_array_multitaper(data_og, sfreq=s_freq, fmin=freq_band[0], fmax=freq_band[1], normalization='length', verbose=False)
        band2features[freq_band].append({
            'frontal2parietal': get_frontal2parietal_gradient(channels, psd),
            'front2back': get_front2back_gradient(channels, psd),
            'left2right': get_left2right_gradient(channels, psd),
            'diff': (psd - psd_og) / (psd + psd_og),
            'label': zg_labels[i]
        })
    psd, freqs = mne.time_frequency.psd_array_multitaper(data_zg, sfreq=s_freq, fmin=8, fmax=26, normalization='length', verbose=False)
    psd_og, freqs_og = mne.time_frequency.psd_array_multitaper(data_og, sfreq=s_freq, fmin=8, fmax=26, normalization='length', verbose=False)
    band2features[(8, 26)].append({
        'diff': (psd - psd_og) / (psd + psd_og),
        'features': extract_features(data_zg, s_freq).flatten(),
        'label': zg_labels[i]
    })
    

num_samples = len(band2features[freq_bands[0]])
frontal2parietal_features = [np.concatenate([band2features[freq_bands[0]][i]['frontal2parietal'], band2features[freq_bands[1]][i]['frontal2parietal']]) for i in range(num_samples)]
front2back_features = [np.concatenate([band2features[freq_bands[0]][i]['front2back'], band2features[freq_bands[1]][i]['front2back']]) for i in range(num_samples)]
left2right_features = [np.concatenate([band2features[freq_bands[0]][i]['left2right'], band2features[freq_bands[1]][i]['left2right']]) for i in range(num_samples)]
# other_features = [np.concatenate([band2features[freq_bands[0]][i]['features'], band2features[freq_bands[1]][i]['features']]) for i in range(num_samples)]
other_features = [band2features[(8, 26)][i]['features'].flatten() for i in range(num_samples)]
diff_features = [band2features[(8, 26)][i]['diff'].flatten() for i in range(num_samples)]
labels = [label2class[x['label']] for x in band2features[freq_bands[0]]]

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=92)

features = []
for i in range(len(labels)):
    features.append(np.concatenate([frontal2parietal_features[i], front2back_features[i], left2right_features[i], diff_features[i], other_features[i]]))
f1_scores = []
for i, (train, test) in enumerate(kf.split(list(range(len(labels))), labels)):
    clf = SVC(kernel='linear', random_state=92)
    X_train = np.array(features)[train]
    X_test = np.array(features)[test]
    y_train = np.array(labels)[train]
    y_test = np.array(labels)[test]
    clf.fit(X_train, y_train)   
    f1_scores.append(f1_score(y_test, clf.predict(X_test), average='macro'))

# print(f'Frequency band: {freq_band}')
print(f'\tF1 score: {np.mean(f1_scores)}, std: {np.std(f1_scores)}')

  0%|          | 0/461 [00:00<?, ?it/s]

	F1 score: 0.40166753184918214, std: 0.06158667388859414
